In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch


In [2]:
from torch.hub import load

In [3]:
model = load('facebookresearch/pytorchvideo', 'x3d_s', pretrained=False)

# Modify final layer
model.blocks[5].proj = torch.nn.Linear(model.blocks[5].proj.in_features, 10)
model.load_state_dict(torch.load("x3d_s_custom_10cls_state.pth"))


Using cache found in C:\Users\rajni/.cache\torch\hub\facebookresearch_pytorchvideo_main


<All keys matched successfully>

In [4]:
import json
import urllib
from pytorchvideo.data.encoded_video import EncodedVideo

from torchvision.transforms import Compose, Lambda
from torchvision.transforms._transforms_video import (
    CenterCropVideo,
    NormalizeVideo,
)
from pytorchvideo.transforms import (
    ApplyTransformToKey,
    ShortSideScale,
    UniformTemporalSubsample
)

e:\synclabs\edgeface\.venv\Lib\site-packages\torchvision\transforms\_functional_video.py:6: UserWarning: The 'torchvision.transforms._functional_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms.functional' module instead.
  warnings.warn(
e:\synclabs\edgeface\.venv\Lib\site-packages\torchvision\transforms\_transforms_video.py:22: UserWarning: The 'torchvision.transforms._transforms_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms' module instead.
  warnings.warn(


In [5]:
device = "cuda"
model = model.eval()
model = model.to(device)

In [6]:
mean = [0.45, 0.45, 0.45]
std = [0.225, 0.225, 0.225]
frames_per_second = 30
model_transform_params  = {
    "x3d_xs": {
        "side_size": 182,
        "crop_size": 182,
        "num_frames": 4,
        "sampling_rate": 12,
    },
    "x3d_s": {
        "side_size": 182,
        "crop_size": 182,
        "num_frames": 13,
        "sampling_rate": 6,
    },
    "x3d_m": {
        "side_size": 256,
        "crop_size": 256,
        "num_frames": 16,
        "sampling_rate": 5,
    }
}

# Get transform parameters based on model
transform_params = model_transform_params["x3d_s"]

# Note that this transform is specific to the slow_R50 model.
transform =  ApplyTransformToKey(
    key="video",
    transform=Compose(
        [
            UniformTemporalSubsample(transform_params["num_frames"]),
            Lambda(lambda x: x/255.0),
            NormalizeVideo(mean, std),
            ShortSideScale(size=transform_params["side_size"]),
            CenterCropVideo(
                crop_size=(transform_params["crop_size"], transform_params["crop_size"])
            )
        ]
    ),
)

# The duration of the input clip is also specific to the model.
clip_duration = (transform_params["num_frames"] * transform_params["sampling_rate"])/frames_per_second

In [7]:
model.blocks[5].proj

Linear(in_features=2048, out_features=10, bias=True)

In [8]:
import torch.nn as nn

# Get input features of the current classification layer
in_features = model.blocks[5].proj.in_features
print("Input features:", in_features)  # Should print 2048

# Replace with new Linear layer for 10 classes
model.blocks[5].proj = nn.Linear(in_features, 10, bias=True)

Input features: 2048


In [9]:
print(model)

Net(
  (blocks): ModuleList(
    (0): ResNetBasicStem(
      (conv): Conv2plus1d(
        (conv_t): Conv3d(3, 24, kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1), bias=False)
        (conv_xy): Conv3d(24, 24, kernel_size=(5, 1, 1), stride=(1, 1, 1), padding=(2, 0, 0), groups=24, bias=False)
      )
      (norm): BatchNorm3d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (activation): ReLU()
    )
    (1): ResStage(
      (res_blocks): ModuleList(
        (0): ResBlock(
          (branch1_conv): Conv3d(24, 24, kernel_size=(1, 1, 1), stride=(1, 2, 2), bias=False)
          (branch2): BottleneckBlock(
            (conv_a): Conv3d(24, 54, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
            (norm_a): BatchNorm3d(54, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (act_a): ReLU()
            (conv_b): Conv3d(54, 54, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1), groups=54, bias=False)
            (nor

In [10]:
import os
from glob import glob
import torch
from torch.utils.data import Dataset
from pytorchvideo.data.encoded_video import EncodedVideo

class VideoDataset(Dataset):
    def __init__(self, base_path, transform, clip_duration, class_map=None):
        self.base_path = base_path
        self.transform = transform
        self.clip_duration = clip_duration
        self.samples = []

        # Create class map
        if class_map is None:
            classes = sorted([d for d in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, d))])
            self.class_map = {cls: idx for idx, cls in enumerate(classes)}
        else:
            self.class_map = class_map

        # Collect video paths
        for cls_name, cls_idx in self.class_map.items():
            class_dir = os.path.join(base_path, cls_name)
            videos = glob(os.path.join(class_dir, "*.mp4"))
            for video in videos:
                self.samples.append((video, cls_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        video = EncodedVideo.from_path(video_path)
        clip = video.get_clip(start_sec=0, end_sec=self.clip_duration)["video"]  # (T, H, W, C)

        # If video is too short
        if clip is None or clip.numel() == 0:
            raise ValueError(f"Empty clip from {video_path}")

        if clip.shape[-1] == 3:
           clip = clip.permute(3, 0, 1, 2)

        # Wrap in dict for ApplyTransformToKey
        sample = {"video": clip}

        if self.transform:
            sample = self.transform(sample)

        return sample["video"], label


In [11]:
from torch.utils.data import DataLoader

base_dataset = "useful_videos"

train_dataset = VideoDataset(
    base_path=os.path.join(base_dataset, "train"),
    transform=transform,
    clip_duration=clip_duration
)

val_dataset = VideoDataset(
    base_path=os.path.join(base_dataset, "val"),
    transform=transform,
    clip_duration=clip_duration,
    class_map=train_dataset.class_map  # Use same mapping
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

print(f"Train dataset size: {len(train_dataset)} videos")
print(f"Validation dataset size: {len(val_dataset)} videos")


Train dataset size: 9358 videos
Validation dataset size: 646 videos


In [12]:
# Get one batch of data
clips, labels = next(iter(train_loader))

print(f"Clips shape: {clips.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Memory per batch: {clips.element_size() * clips.numel() / 1e6:.2f} MB")


Clips shape: torch.Size([4, 3, 13, 182, 182])
Labels shape: torch.Size([4])
Memory per batch: 20.67 MB


In [13]:
# Select the duration of the clip to load by specifying the start and end duration
# The start_sec should correspond to where the action occurs in the video
start_sec = 0
end_sec = start_sec + clip_duration

video_path = "useful_videos/train/slapping/_tOe654jyY0_000003_000013.mp4"
# Initialize an EncodedVideo helper class and load the video
video = EncodedVideo.from_path(video_path)

# Load the desired clip
video_data = video.get_clip(start_sec=start_sec, end_sec=end_sec)

# Apply a transform to normalize the video input
video_data = transform(video_data)

# Move the inputs to the desired device
inputs = video_data["video"]
inputs = inputs.to(device)

In [14]:
base_folder = "useful_videos"
video_extensions = (".mp4", ".avi", ".mov", ".mkv", ".wmv")


In [15]:
# def get_video_lengths(base_folder, video_extensions=(".mp4", ".avi", ".mov", ".mkv")):
#     """
#     Recursively scans the base_folder for videos, calculates their lengths,
#     and returns a list of (video_path, duration_seconds).
#     Also plots a histogram of video durations with 0.4-second bins.
#     """
#     video_paths = []
#     for root, dirs, files in os.walk(base_folder):
#         for file in files:
#             if file.lower().endswith(video_extensions):
#                 video_paths.append(os.path.join(root, file))
    
#     print(f"Found {len(video_paths)} videos in '{base_folder}'.")
    
#     video_lengths = []
#     video_info = []

#     for video in video_paths:
#         cap = cv2.VideoCapture(video)
#         if not cap.isOpened():
#             print(f"Failed to open video: {video}")
#             continue

#         frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
#         fps = cap.get(cv2.CAP_PROP_FPS)
#         duration = frame_count / fps if fps > 0 else 0
#         video_lengths.append(duration)
#         video_info.append((video, duration))
#         cap.release()

#     if video_lengths:
#         max_length = max(video_lengths)
#         bins = int(max_length / 0.4) + 1
#         plt.hist(video_lengths, bins=bins, edgecolor='black')
#         plt.xlabel('Video Length (seconds)')
#         plt.ylabel('Number of Videos')
#         plt.title('Video Length Distribution (0.4s bins)')
#         plt.grid(True)
#         plt.show()
#     else:
#         print("No valid video lengths found.")
    
#     return video_info

In [16]:
import os
import cv2

In [17]:
model.blocks[5]

ResNetBasicHead(
  (pool): ProjectedPool(
    (pre_conv): Conv3d(192, 432, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
    (pre_norm): BatchNorm3d(432, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (pre_act): ReLU()
    (pool): AvgPool3d(kernel_size=(np.int64(13), 5, 5), stride=1, padding=0)
    (post_conv): Conv3d(432, 2048, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
    (post_act): ReLU()
  )
  (dropout): Dropout(p=0.5, inplace=False)
  (proj): Linear(in_features=2048, out_features=10, bias=True)
  (output_pool): AdaptiveAvgPool3d(output_size=1)
)

In [18]:
import torch
import torch.nn as nn
import torch.optim as optim

# Assume 'model' is your modified x3d_s model

# 1. Freeze all parameters
for param in model.parameters():
    param.requires_grad = False

# 2. Unfreeze the last two blocks and final proj layer
for param in model.blocks[5].proj.parameters():
    param.requires_grad = True

# 3. Define optimizer with different learning rates
# Separate proj params from the rest of block 5
head_params = list(model.blocks[5].proj.parameters())
other_head_params = [p for n, p in model.blocks[5].named_parameters() if "proj" not in n]

optimizer = optim.AdamW([
    {"params": other_head_params, "lr": 1e-4},
    {"params": head_params, "lr": 1e-3}
], weight_decay=1e-4)
# (Optional) Define a scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.1)


In [19]:
import mlflow.pytorch

In [ ]:
# --- Setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
criterion = nn.CrossEntropyLoss()
epochs = 100
save_dir = "checkpoints/xds"
os.makedirs(save_dir, exist_ok=True)

: 

In [ ]:
mlflow.set_experiment("x3d_training")
with mlflow.start_run():
    # Log parameters
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("optimizer", optimizer.__class__.__name__)
    mlflow.log_param("learning_rates", [g["lr"] for g in optimizer.param_groups])
    mlflow.log_param("scheduler", "StepLR (step_size=100, gamma=0.1)")

    for epoch in range(1, epochs + 1):
        # ---- Training ----
        model.train()
        total_train_loss = 0
        total_train_correct = 0
        total_train_samples = 0

        for clips, labels in train_loader:
            clips, labels = clips.to(device), labels.to(device)
            optimizer.zero_grad()

            outputs = model(clips)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item() * clips.size(0)
            total_train_correct += (outputs.argmax(1) == labels).sum().item()
            total_train_samples += labels.size(0)

        train_loss = total_train_loss / total_train_samples
        train_acc = total_train_correct / total_train_samples

        # ---- Validation ----
        model.eval()
        total_val_loss = 0
        total_val_correct = 0
        total_val_samples = 0
        with torch.no_grad():
            for clips, labels in val_loader:
                clips, labels = clips.to(device), labels.to(device)
                outputs = model(clips)
                loss = criterion(outputs, labels)

                total_val_loss += loss.item() * clips.size(0)
                total_val_correct += (outputs.argmax(1) == labels).sum().item()
                total_val_samples += labels.size(0)

        val_loss = total_val_loss / total_val_samples
        val_acc = total_val_correct / total_val_samples

        current_lr = scheduler.get_last_lr()
        print(f"Epoch [{epoch}/{epochs}] "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | "
              f"LR: {current_lr}")

        # ---- MLflow Logging ----
        mlflow.log_metrics({
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
            "lr": current_lr[0]
        }, step=epoch)

        # ---- Scheduler Step ----
        scheduler.step()

        # ---- Save model every 10 epochs ----
        if epoch % 10 == 0:
            save_path = os.path.join(save_dir, f"x3d_epoch_{epoch}.pth")
            torch.save(model.state_dict(), save_path)
            mlflow.log_artifact(save_path)
            print(f"Model checkpoint saved to {save_path}")

    # Log final model to MLflow
    mlflow.pytorch.log_model(model, "final_x3d_model")
    print("Final model logged to MLflow")